In [5]:
import os
import sys
import subprocess
import pandas as pd
from IPython.display import display

RULES_DIR = "./data/rules"

In [6]:
sections_df = pd.read_csv("./data/index_diachronica_sections.csv", dtype={"index": str, "name": str, "rule_count": int})

rules_df = sections_df[sections_df["rule_count"] > 0].copy()
rules_df['asca_rule_file'] = rules_df['index'].apply(lambda x: f"{x}.rsca")
rules_df['brassica_rule_file'] = rules_df['index'].apply(lambda x: f"{x}.bsc")

display(rules_df.head(3))

asca_rule_files = rules_df['asca_rule_file'].tolist()
brassica_rule_files = rules_df['brassica_rule_file'].tolist()


,index,name,rule_count,asca_rule_file,brassica_rule_file
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,11,6.1.rsca,6.1.bsc
2,6.1.1,Proto-Omotic to North Omotic,18,6.1.1.rsca,6.1.1.bsc
3,6.1.1.1,North Omotic to Bench,9,6.1.1.1.rsca,6.1.1.1.bsc


In [7]:
# run asca-rust to validate rules

asca_results = []
asca_word_file = "./data/words/asca/weirdness_0.5.wsca"

def run_asca(asca_word_file, rule_file):
    cmd = f"~/.cargo/bin/asca run {asca_word_file} -r {RULES_DIR}/asca/{rule_file}"

    result = {"rule": rule_file, "returncode": 0, "error": ""}

    try:
        output = subprocess.run(cmd, capture_output=True, timeout=10, shell=True, text=True)
        output.check_returncode()

    except subprocess.CalledProcessError as exc:
        result["returncode"] = exc.returncode 
        result["error"] = exc.output.replace("\n", "\\n")
    except subprocess.TimeoutExpired as exc:
        result["returncode"] = 124
        result["error"] = exc.output.decode("utf-8").replace("\n", "\\n")

    return result

for rule_file in asca_rule_files[:10]:
    result = run_asca(asca_word_file, rule_file)
    asca_results.append(result)

asca_results_df = pd.DataFrame(asca_results)

asca_results_df.to_csv("./data/asca_results.csv", index=False)

asca_results_df[asca_results_df["returncode"] != 0].head()

,rule,returncode,error


In [ ]:
# run Brassica to validate rules

brassica_results = []
brassica_word_file = "./data/words/brassica/weirdness_0.5.lex"

def run_brassica(brassica_word_file, rule_file):
    cmd = f"brassica {RULES_DIR}/brassica/{rule_file} -i {brassica_word_file}"

    result = {"rule": rule_file, "returncode": 0, "error": ""}

    try:
        print(cmd)
        output = subprocess.run(cmd, capture_output=True, timeout=10, shell=True, text=True)
        print(output.stdout)
        output.check_returncode()

    except subprocess.CalledProcessError as exc:
        result["returncode"] = exc.returncode 
        result["error"] = exc.output.replace("\n", "\\n")
    except subprocess.TimeoutExpired as exc:
        result["returncode"] = 124
        result["error"] = exc.output.decode("utf-8").replace("\n", "\\n")

    return result

for rule_file in brassica_rule_files[:10]:
    result = run_brassica(brassica_word_file, rule_file)
    brassica_results.append(result)

brassica_results_df = pd.DataFrame(brassica_results)

brassica_results_df.to_csv("./data/brassica_results.csv", index=False)

brassica_results_df[brassica_results_df["returncode"] != 0].head()

brassica ./data/rules/brassica/6.1.bsc -i ./data/words/brassica/weirdness_0.5.lex
CompletedProcess(args='brassica ./data/rules/brassica/6.1.bsc -i ./data/words/brassica/weirdness_0.5.lex', returncode=0, stdout='4:12:\n  |\n4 | dʒ / tʃ / ʃ\n  |            ^\nunexpected newline\nexpecting "%(", \'#\', \'$\', \'%\', \'(\', \'*\', \'>\', \'@\', \'[\', \'^\', \'_\', or \'~\'\n\n', stderr='')
brassica ./data/rules/brassica/6.1.1.bsc -i ./data/words/brassica/weirdness_0.5.lex
CompletedProcess(args='brassica ./data/rules/brassica/6.1.1.bsc -i ./data/words/brassica/weirdness_0.5.lex', returncode=0, stdout='5:12:\n  |\n5 | e / i / #l_{P,C[+voiced]}\n  |            ^\nunexpected \'{\'\nexpecting "%(", "-1", "->", "-?", "-??", "-ltr", "-no", "-rtl", "-x", "//", "categories", "extra", "filter", "new", "report", \'#\', \'%\', \'(\', \'*\', \'/\', \'>\', \'@\', \'[\', \'^\', \'→\', or end of input\n\n', stderr='')
brassica ./data/rules/brassica/6.1.1.1.bsc -i ./data/words/brassica/weirdness_0.5.lex
C

,rule,returncode,error
